# Phase 1: Model Prototyping

This notebook is for interactively developing and testing the core components of our transformer model. Following our agreed workflow, we will:

1.  **Implement** model `nn.Module` classes in the `src/` directory (e.g., `src/model.py`).
2.  **Import** those classes here in the notebook.
3.  **Test** them with sample data to verify their correctness (e.g., checking tensor shapes, data flow, and outputs).

Let's start by loading our dataset and creating a simple character-level tokenizer.

## 1. Data Loading and Tokenization

In [1]:
# Read the dataset
with open('../data/tinyshakespeare/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
print("Length of dataset in characters: ", len(text))

Length of dataset in characters:  1115394


In [3]:
print(text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


Now, we'll create a simple character-level tokenizer.

In [4]:
# Get all the unique characters in the text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(f"Vocabulary size: {vocab_size}")

for char in chars:
    print([char, ord(char)])


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary size: 65
['\n', 10]
[' ', 32]
['!', 33]
['$', 36]
['&', 38]
["'", 39]
[',', 44]
['-', 45]
['.', 46]
['3', 51]
[':', 58]
[';', 59]
['?', 63]
['A', 65]
['B', 66]
['C', 67]
['D', 68]
['E', 69]
['F', 70]
['G', 71]
['H', 72]
['I', 73]
['J', 74]
['K', 75]
['L', 76]
['M', 77]
['N', 78]
['O', 79]
['P', 80]
['Q', 81]
['R', 82]
['S', 83]
['T', 84]
['U', 85]
['V', 86]
['W', 87]
['X', 88]
['Y', 89]
['Z', 90]
['a', 97]
['b', 98]
['c', 99]
['d', 100]
['e', 101]
['f', 102]
['g', 103]
['h', 104]
['i', 105]
['j', 106]
['k', 107]
['l', 108]
['m', 109]
['n', 110]
['o', 111]
['p', 112]
['q', 113]
['r', 114]
['s', 115]
['t', 116]
['u', 117]
['v', 118]
['w', 119]
['x', 120]
['y', 121]
['z', 122]


In [5]:
# Create a mapping from characters to integers (stoi) and integers to characters (itos)
stoi: dict[str, int] = {ch: i for i, ch in enumerate(chars)}
itos: dict[int, str] = {i: ch for i, ch in enumerate(chars)}

# Encoder:
def encode(s: str) -> list[int]:
	return [stoi[c] for c in s]

# Decoder:
def decode(l: list[int]) -> str:
	return ''.join([itos[i] for i in l])

# Test the tokenizer
test_string = "hello world"
encoded_string = encode(test_string)
decoded_string = decode(encoded_string)

print(f"Original: '{test_string}'")
print(f"Encoded: {encoded_string}")
print(f"Decoded: '{decoded_string}'")

Original: 'hello world'
Encoded: [46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
Decoded: 'hello world'


With the tokenizer ready, we can now convert our entire dataset into a `torch.Tensor`.

In [6]:
import torch

# Encode the entire text dataset and store it in a torch.Tensor
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100]) # The first 100 characters we saw earlier, now as integers

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


## 2. Splitting Data into Training and Validation Sets

We'll use the first 90% of the data for training and the remaining 10% for validation.

In [7]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

print(f"Training data length: {len(train_data)}")
print(f"Validation data length: {len(val_data)}")

Training data length: 1003854
Validation data length: 111540


## 3. Testing the Model Components

In [ ]:
import torch
from torch import Tensor
import torch.nn as nn

torch.manual_seed(0) # pyright: ignore[reportUnknownMemberType]

In [9]:
import importlib
import model as model_module
importlib.reload(model_module)

<module 'model' from '/Users/williamboan/Documents/Career Transition - Projects/Transformer From Scratch/transformer-from-scratch/src/model.py'>

In [10]:
from config import (
    block_size as DEFAULT_BLOCK_SIZE,
    n_embd as DEFAULT_N_EMBD,
    dropout as DEFAULT_DROPOUT,
)

print("Default block_size:", DEFAULT_BLOCK_SIZE)
print("Default n_embd:", DEFAULT_N_EMBD)
print("Default dropout:", DEFAULT_DROPOUT)

Default block_size: 8
Default n_embd: 32
Default dropout: 0.0


### Head

In [11]:
import model as model_module
importlib.reload(model_module)

Head = model_module.Head

In [12]:
# Basic forward pass + shape checks
B, T, C = 2, min(5, DEFAULT_BLOCK_SIZE), DEFAULT_N_EMBD
head_size = 16

head = Head(head_size, n_embd=DEFAULT_N_EMBD, block_size=DEFAULT_BLOCK_SIZE)
x = torch.randn(B, T, C)

out: Tensor = head(x)

print("x.shape:", x.shape)
print("out.shape:", out.shape)
assert out.shape == (B, T, head_size)

x.shape: torch.Size([2, 5, 32])
out.shape: torch.Size([2, 5, 16])


In [13]:
# Causality check
# Changing the LAST token should NOT affect outputs at earlier timesteps.
with torch.no_grad():
    x2 = x.clone()
    x2[:, -1, :] += 999.0  # big perturbation at the final position

    out1: Tensor = head(x)
    out2: Tensor = head(x2)

    max_diff_earlier = (out1[:, :-1, :] - out2[:, :-1, :]).abs().max().item()
    max_diff_last = (out1[:, -1, :] - out2[:, -1, :]).abs().max().item()

print("max_diff earlier positions:", max_diff_earlier)
print("max_diff last position:", max_diff_last)

assert max_diff_earlier < 1e-6, "Causal mask appears broken: earlier outputs changed."
assert max_diff_last > 1e-6, "Unexpected: last position did not change after perturbation."

max_diff earlier positions: 0.0
max_diff last position: 1461.4119873046875


In [14]:
# Gradient flow check
head = Head(head_size=head_size, n_embd=DEFAULT_N_EMBD, block_size=DEFAULT_BLOCK_SIZE)
x = torch.randn(B, T, C, requires_grad=True)

out: Tensor = head(x)
loss: Tensor = out.pow(2).mean()
loss.backward() # type: ignore

assert x.grad is not None
assert head.key.weight.grad is not None

print("x.grad max abs:", x.grad.abs().max().item())
print("key.weight grad max abs:", head.key.weight.grad.abs().max().item())

assert torch.isfinite(x.grad).all()
assert torch.isfinite(head.key.weight.grad).all()

x.grad max abs: 0.017195824533700943
key.weight grad max abs: 0.010610200464725494


### MultiHeadAttention

In [15]:
importlib.reload(model_module)
MultiHeadAttention = model_module.MultiHeadAttention

# --- Hyperparameters for the test ---
n_head = 4
B, T, C = 2, min(5, DEFAULT_BLOCK_SIZE), DEFAULT_N_EMBD  # (Batch, Time, Channels)

# --- Test Forward Pass ---
mha = MultiHeadAttention(
    n_head=n_head,
    n_embd=DEFAULT_N_EMBD,
    block_size=DEFAULT_BLOCK_SIZE,
    dropout=DEFAULT_DROPOUT,
)
x = torch.randn(B, T, C)
out: Tensor = mha(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)
assert out.shape == (B, T, C), "Output shape is incorrect!"
print("Forward pass and shape check successful.")

Input shape: torch.Size([2, 5, 32])
Output shape: torch.Size([2, 5, 32])
Forward pass and shape check successful.


In [16]:
# --- Test Gradient Flow ---
mha = MultiHeadAttention(
    n_head=n_head,
    n_embd=DEFAULT_N_EMBD,
    block_size=DEFAULT_BLOCK_SIZE,
    dropout=DEFAULT_DROPOUT,
)
x = torch.randn(B, T, C, requires_grad=True)

out: Tensor = mha(x)
loss: Tensor = out.pow(2).mean()
loss.backward()  # type: ignore

head1: Head = mha.heads[0]  # type: ignore

assert x.grad is not None
assert mha.proj.weight.grad is not None
assert head1.key.weight.grad is not None

print("x.grad max abs:", x.grad.abs().max().item())
print("proj.weight grad max abs:", mha.proj.weight.grad.abs().max().item())
print("head[0].key.weight grad max abs:", head1.key.weight.grad.abs().max().item())

assert torch.isfinite(x.grad).all()
assert torch.isfinite(mha.proj.weight.grad).all()
assert torch.isfinite(head1.key.weight.grad).all()
print("Gradient flow check successful.")

x.grad max abs: 0.005334151443094015
proj.weight grad max abs: 0.016739774495363235
head[0].key.weight grad max abs: 0.001678013359196484
Gradient flow check successful.


### FeedForward

In [17]:
# Reload the model module to pick up the new FeedForward class
importlib.reload(model_module)
FeedForward = model_module.FeedForward

# --- Test Forward Pass ---
ffwd = FeedForward(n_embd=DEFAULT_N_EMBD, dropout=DEFAULT_DROPOUT)
x = torch.randn(B, T, C)
out: Tensor = ffwd(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)
assert out.shape == x.shape, "Output shape is incorrect!"
print("Forward pass and shape check successful.")

Input shape: torch.Size([2, 5, 32])
Output shape: torch.Size([2, 5, 32])
Forward pass and shape check successful.


In [18]:
# --- Test Gradient Flow ---
ffwd = FeedForward(n_embd=DEFAULT_N_EMBD, dropout=DEFAULT_DROPOUT)
x = torch.randn(B, T, C, requires_grad=True)

out: Tensor = ffwd(x)
loss: Tensor = out.pow(2).mean()
loss.backward() # type: ignore

# Accessing layers inside nn.Sequential
linear1: nn.Linear = ffwd.net[0] # type: ignore
linear2: nn.Linear = ffwd.net[2] # type: ignore

assert x.grad is not None
assert linear1.weight.grad is not None
assert linear2.weight.grad is not None

print("x.grad max abs:", x.grad.abs().max().item())
print("linear1.weight grad max abs:", linear1.weight.grad.abs().max().item())
print("linear2.weight grad max abs:", linear2.weight.grad.abs().max().item())

assert torch.isfinite(x.grad).all()
assert torch.isfinite(linear1.weight.grad).all()
assert torch.isfinite(linear2.weight.grad).all()
print("Gradient flow check successful.")

x.grad max abs: 0.0020266242790967226
linear1.weight grad max abs: 0.007181484252214432
linear2.weight grad max abs: 0.021599501371383667
Gradient flow check successful.


### Block

In [19]:
# Reload the model module to pick up the new Block class
importlib.reload(model_module)
Block = model_module.Block

# --- Test Forward Pass ---
block = Block(
    n_head=n_head,
    n_embd=DEFAULT_N_EMBD,
    block_size=DEFAULT_BLOCK_SIZE,
    dropout=DEFAULT_DROPOUT,
)
x = torch.randn(B, T, C)
out: Tensor = block(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)
assert out.shape == x.shape, "Output shape is incorrect!"
print("Forward pass and shape check successful.")

Input shape: torch.Size([2, 5, 32])
Output shape: torch.Size([2, 5, 32])
Forward pass and shape check successful.


In [20]:
# --- Test Gradient Flow ---
block = Block(
    n_head=n_head,
    n_embd=DEFAULT_N_EMBD,
    block_size=DEFAULT_BLOCK_SIZE,
    dropout=DEFAULT_DROPOUT,
)
x = torch.randn(B, T, C, requires_grad=True)

out: Tensor = block(x)
loss: Tensor = out.pow(2).mean()
loss.backward()  # type: ignore

ffwd_linear1: nn.Linear = block.ffwd.net[0]  # type: ignore

assert x.grad is not None
assert block.mha.proj.weight.grad is not None
assert ffwd_linear1.weight.grad is not None
assert block.ln1.weight.grad is not None

print("x.grad max abs:", x.grad.abs().max().item())
print("sa.proj.weight grad max abs:", block.mha.proj.weight.grad.abs().max().item())
print("ffwd.net[0].weight grad max abs:", ffwd_linear1.weight.grad.abs().max().item())
print("ln1.weight grad max abs:", block.ln1.weight.grad.abs().max().item())

assert torch.isfinite(x.grad).all()
assert torch.isfinite(block.mha.proj.weight.grad).all()
print("Gradient flow check successful.")

x.grad max abs: 0.025972044095396996
sa.proj.weight grad max abs: 0.04614727571606636
ffwd.net[0].weight grad max abs: 0.0246879979968071
ln1.weight grad max abs: 0.018274245783686638
Gradient flow check successful.


### Transformer (Full Model)

In [21]:
# Reload the model module to pick up the final Transformer class
importlib.reload(model_module)
Transformer = model_module.Transformer

# --- Hyperparameters for the test ---
n_layer = 2 # Number of transformer blocks
n_head = 4
B, T, C = 2, DEFAULT_BLOCK_SIZE, DEFAULT_N_EMBD

# --- Test Forward Pass & Loss Calculation ---
model = Transformer(vocab_size=vocab_size, n_layer=n_layer, n_head=n_head)

# Create dummy input and target tensors
idx = torch.randint(0, vocab_size, (B, T))
targets = torch.randint(0, vocab_size, (B, T))

logits: Tensor
loss: Tensor
logits, loss = model(idx, targets)

print("--- Shapes ---")
print("Input `idx` shape:", idx.shape)
print("Output `logits` shape:", logits.shape)
print("\n--- Loss ---")
print("Calculated loss:", loss.item()) # type: ignore

# --- Assertions ---
assert logits.shape == (B, T, vocab_size), "Logits shape is incorrect!"
assert loss is not None, "Loss should not be None when targets are provided."
assert loss.ndim == 0, "Loss should be a scalar tensor."
assert torch.isfinite(loss), "Loss is not a finite number."
print("\nForward pass, shape, and loss checks successful.")

--- Shapes ---
Input `idx` shape: torch.Size([2, 8])
Output `logits` shape: torch.Size([2, 8, 65])

--- Loss ---
Calculated loss: 4.1812357902526855

Forward pass, shape, and loss checks successful.


In [22]:
# --- Test Gradient Flow ---
model = Transformer(vocab_size=vocab_size, n_layer=n_layer, n_head=n_head)
idx = torch.randint(0, vocab_size, (B, T))
targets = torch.randint(0, vocab_size, (B, T))

logits, loss = model(idx, targets)
loss.backward() # type: ignore

block1: Block = model.blocks[0] # type: ignore

# Check gradients in key parts of the model
assert model.token_embedding_table.weight.grad is not None
assert block1.mha.proj.weight.grad is not None
assert model.lm_head.weight.grad is not None

print("token_embedding_table grad max abs:", model.token_embedding_table.weight.grad.abs().max().item())
print("lm_head grad max abs:", model.lm_head.weight.grad.abs().max().item())
assert torch.isfinite(model.token_embedding_table.weight.grad).all()
assert torch.isfinite(model.lm_head.weight.grad).all()
print("\nGradient flow check successful.")

token_embedding_table grad max abs: 0.15235298871994019
lm_head grad max abs: 0.21636717021465302

Gradient flow check successful.


### Inference with `generate`

In [23]:
# Reload the model to ensure we have the generate method
importlib.reload(model_module)
Transformer = model_module.Transformer

model = Transformer(vocab_size=vocab_size, n_layer=n_layer, n_head=n_head)

# Create a starting context: a tensor of shape (1, 1) with a newline character
# A newline is a good "neutral" starting point.
start_context = torch.zeros((1, 1), dtype=torch.long) # 0 is the index for newline

# Generate 100 new tokens
generated_indices = model.generate(start_context, max_new_tokens=100)

# The output is a (1, 101) tensor. We need the first batch's indices.
generated_sequence: list[int] = generated_indices[0].tolist() # type: ignore

# Decode the indices back to text
generated_text = decode(generated_sequence)

print("--- Generated Text (Untrained Model) ---")
print(generated_text)

--- Generated Text (Untrained Model) ---

qrl3PlfN;F.glhADBGofvZ'!j!w'vD:xd-QZD.mh ;L XDpQMyTW!FfCVo$wbveBq&hTASZlZPpw,qPy;;Uvfnnn,NwewQoUoF&m


NOTE: As expected, the output is random nonsense. The model has not learned any patterns yet.

In [ ]:
# 